# Guía de Creación Masiva de Municipios
Esta guía demuestra cómo usar correctamente el endpoint `POST /api/catalog/municipios/bulk` para crear municipios de forma masiva, incluyendo manejo de errores y validaciones.

## Error Identificado
Tu request actual falla porque tiene un string simple sin `defaultDepartamentoId`:
```json
{
  "municipios": [
    "string",  // ❌ Falta defaultDepartamentoId para strings
    {
      "nombre_municipio": "Bogotá D.C.",
      "codigo_dane": "11001", 
      "id_departamento": 1
    }
  ]
}
```

# Import Required Libraries
Importar las librerías necesarias para manejo de datos, validación y operaciones de API.

In [ ]:
# Import Required Libraries
import requests
import json
from datetime import datetime
from typing import List, Dict, Union, Optional
import logging

# Configurar logging para depuración
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Define Data Structures
Definir las estructuras de datos para municipios con los campos apropiados como nombre_municipio, codigo_dane, y id_departamento.

In [ ]:
# Define Data Structures
class MunicipioData:
    """Estructura de datos para un municipio"""
    def __init__(self, nombre_municipio: str, codigo_dane: str = None, id_departamento: int = None):
        self.nombre_municipio = nombre_municipio
        self.codigo_dane = codigo_dane
        self.id_departamento = id_departamento
    
    def to_dict(self) -> Dict:
        """Convertir a diccionario para JSON"""
        result = {"nombre_municipio": self.nombre_municipio}
        if self.codigo_dane:
            result["codigo_dane"] = self.codigo_dane
        if self.id_departamento:
            result["id_departamento"] = self.id_departamento
        return result

# Estructura del request body correcto
class BulkMunicipiosRequest:
    """Estructura del request para creación masiva"""
    def __init__(self, municipios: List[Union[str, Dict]], default_departamento_id: int = None):
        self.municipios = municipios
        self.default_departamento_id = default_departamento_id
    
    def to_dict(self) -> Dict:
        """Convertir a diccionario para el request"""
        result = {"municipios": self.municipios}
        if self.default_departamento_id:
            result["defaultDepartamentoId"] = self.default_departamento_id
        return result

# Ejemplos de datos válidos
ejemplo_municipios_completos = [
    {
        "nombre_municipio": "Bogotá D.C.",
        "codigo_dane": "11001",
        "id_departamento": 1
    },
    {
        "nombre_municipio": "Medellín",
        "codigo_dane": "05001", 
        "id_departamento": 2
    }
]

ejemplo_municipios_mixtos = [
    "Municipio Simple 1",  # String simple
    "Municipio Simple 2",  # String simple
    {
        "nombre_municipio": "Cali", 
        "codigo_dane": "76001",
        "id_departamento": 3
    }
]

# Validate Municipio Data
Crear funciones de validación para verificar si los municipios tienen los campos requeridos y tipos de datos apropiados antes de la creación masiva.

In [ ]:
# Validate Municipio Data
def validate_municipio_object(municipio: Dict) -> tuple[bool, str]:
    """Validar un objeto municipio individual"""
    if not isinstance(municipio, dict):
        return False, "Municipio debe ser un diccionario"
    
    # Verificar nombre requerido
    if "nombre_municipio" not in municipio or not municipio["nombre_municipio"]:
        return False, "nombre_municipio es requerido"
    
    # Validar tipos de datos
    if not isinstance(municipio["nombre_municipio"], str):
        return False, "nombre_municipio debe ser string"
    
    if "codigo_dane" in municipio and municipio["codigo_dane"] is not None:
        if not isinstance(municipio["codigo_dane"], str):
            return False, "codigo_dane debe ser string"
    
    if "id_departamento" in municipio and municipio["id_departamento"] is not None:
        if not isinstance(municipio["id_departamento"], int):
            return False, "id_departamento debe ser entero"
    
    return True, "Válido"

def validate_municipios_array(municipios: List[Union[str, Dict]]) -> tuple[bool, List[str]]:
    """Validar array completo de municipios"""
    errors = []
    
    if not isinstance(municipios, list):
        return False, ["municipios debe ser un array"]
    
    if len(municipios) == 0:
        return False, ["municipios no puede estar vacío"]
    
    for i, municipio in enumerate(municipios):
        if isinstance(municipio, str):
            # Strings son válidos, se manejan con defaultDepartamentoId
            if not municipio.strip():
                errors.append(f"Municipio {i+1}: String vacío no válido")
        elif isinstance(municipio, dict):
            is_valid, error_msg = validate_municipio_object(municipio)
            if not is_valid:
                errors.append(f"Municipio {i+1}: {error_msg}")
        else:
            errors.append(f"Municipio {i+1}: Debe ser string o objeto")
    
    return len(errors) == 0, errors

def validate_bulk_request(request_data: Dict) -> tuple[bool, List[str]]:
    """Validar request completo de creación masiva"""
    errors = []
    
    # Verificar estructura básica
    if "municipios" not in request_data:
        return False, ["Campo 'municipios' es requerido"]
    
    # Validar array de municipios
    is_valid, municipio_errors = validate_municipios_array(request_data["municipios"])
    if not is_valid:
        errors.extend(municipio_errors)
    
    # Verificar si hay strings sin defaultDepartamentoId
    has_strings = any(isinstance(m, str) for m in request_data["municipios"])
    if has_strings and "defaultDepartamentoId" not in request_data:
        errors.append("defaultDepartamentoId es requerido cuando hay strings en municipios")
    
    # Validar defaultDepartamentoId si existe
    if "defaultDepartamentoId" in request_data:
        if not isinstance(request_data["defaultDepartamentoId"], int):
            errors.append("defaultDepartamentoId debe ser entero")
    
    return len(errors) == 0, errors

# Ejemplo de validación
request_original = {
    "municipios": [
        "string",
        {
            "nombre_municipio": "Bogotá D.C.",
            "codigo_dane": "11001",
            "id_departamento": 1
        }
    ]
}

print("🔍 Validando request original:")
is_valid, errors = validate_bulk_request(request_original)
print(f"Válido: {is_valid}")
if errors:
    print("Errores encontrados:")
    for error in errors:
        print(f"  - {error}")

# Handle Bulk Creation Errors
Implementar manejo de errores para operaciones de creación masiva, incluyendo parsing de respuestas de error y extracción de mensajes significativos.

In [ ]:
# Handle Bulk Creation Errors
class BulkCreationError(Exception):
    """Excepción personalizada para errores de creación masiva"""
    def __init__(self, message: str, code: str = None, details: str = None):
        self.message = message
        self.code = code
        self.details = details
        super().__init__(message)

def parse_api_error_response(response: requests.Response) -> BulkCreationError:
    """Parsear respuesta de error de la API"""
    try:
        error_data = response.json()
        
        if "error" in error_data:
            error_info = error_data["error"]
            return BulkCreationError(
                message=error_info.get("message", "Error desconocido"),
                code=error_info.get("code", "UNKNOWN_ERROR"),
                details=error_info.get("details", "")
            )
        else:
            return BulkCreationError(
                message=error_data.get("message", "Error en la API"),
                code="API_ERROR"
            )
    except json.JSONDecodeError:
        return BulkCreationError(
            message=f"Error HTTP {response.status_code}: {response.text}",
            code="HTTP_ERROR"
        )

def handle_validation_errors(errors: List[str]) -> None:
    """Manejar errores de validación antes de enviar request"""
    if errors:
        print("❌ Errores de validación encontrados:")
        for i, error in enumerate(errors, 1):
            print(f"  {i}. {error}")
        
        print("\n💡 Sugerencias de corrección:")
        
        if any("defaultDepartamentoId" in error for error in errors):
            print("  - Agregar defaultDepartamentoId para manejar strings simples")
        
        if any("nombre_municipio" in error for error in errors):
            print("  - Verificar que todos los objetos tengan nombre_municipio válido")
        
        if any("array" in error.lower() for error in errors):
            print("  - Asegurar que municipios sea un array no vacío")

def demonstrate_error_handling():
    """Demostrar manejo de diferentes tipos de errores"""
    
    # Error 1: Falta defaultDepartamentoId
    print("🧪 Ejemplo 1: Error por falta de defaultDepartamentoId")
    error_request_1 = {
        "municipios": ["string", {"nombre_municipio": "Test"}]
    }
    is_valid, errors = validate_bulk_request(error_request_1)
    handle_validation_errors(errors)
    
    print("\n" + "="*50 + "\n")
    
    # Error 2: Datos inválidos
    print("🧪 Ejemplo 2: Error por datos inválidos")
    error_request_2 = {
        "municipios": [
            {"codigo_dane": "123"},  # Falta nombre_municipio
            123  # Tipo inválido
        ],
        "defaultDepartamentoId": "invalid"  # Tipo inválido
    }
    is_valid, errors = validate_bulk_request(error_request_2)
    handle_validation_errors(errors)

demonstrate_error_handling()

# Implement Default Department Assignment
Crear funcionalidad para asignar IDs de departamento por defecto a entradas de municipio que son solo strings usando options.defaultDepartamentoId.

In [ ]:
# Implement Default Department Assignment
def create_corrected_request_body(municipios: List[Union[str, Dict]], default_departamento_id: int) -> Dict:
    """Crear request body correcto con defaultDepartamentoId"""
    return {
        "municipios": municipios,
        "defaultDepartamentoId": default_departamento_id
    }

def preview_municipio_processing(municipios: List[Union[str, Dict]], default_departamento_id: int = None):
    """Previsualizar cómo se procesarán los municipios"""
    print("🔍 Vista previa del procesamiento de municipios:")
    print(f"📋 Total de municipios: {len(municipios)}")
    print(f"🏛️  Departamento por defecto: {default_departamento_id}")
    print()
    
    for i, municipio in enumerate(municipios, 1):
        if isinstance(municipio, str):
            print(f"  {i}. STRING: '{municipio}'")
            print(f"     → Se convertirá a: {{")
            print(f"         'nombre_municipio': '{municipio}',")
            print(f"         'id_departamento': {default_departamento_id}")
            print(f"       }}")
        else:
            print(f"  {i}. OBJETO: {municipio}")
            if "id_departamento" not in municipio or municipio["id_departamento"] is None:
                print(f"     → Se agregará id_departamento: {default_departamento_id}")
            else:
                print(f"     → Mantiene su id_departamento: {municipio['id_departamento']}")
        print()

# Corrección del request original
print("🔧 CORRECCIÓN DEL REQUEST ORIGINAL")
print("="*50)

municipios_originales = [
    "string",
    {
        "nombre_municipio": "Bogotá D.C.",
        "codigo_dane": "11001",
        "id_departamento": 1
    }
]

# Obtener primer departamento disponible (simulado)
default_dept_id = 1  # Amazonas según nuestras pruebas

print("📝 Request original (INCORRECTO):")
request_incorrecto = {"municipios": municipios_originales}
print(json.dumps(request_incorrecto, indent=2, ensure_ascii=False))

print("\n❌ Problema: Falta defaultDepartamentoId para el string 'string'")

print("\n✅ Request corregido:")
request_corregido = create_corrected_request_body(municipios_originales, default_dept_id)
print(json.dumps(request_corregido, indent=2, ensure_ascii=False))

print("\n🔍 Vista previa del procesamiento:")
preview_municipio_processing(municipios_originales, default_dept_id)

# Create Success Response Handler
Construir manejadores de respuesta para operaciones exitosas de creación masiva y formatear mensajes de éxito apropiados.

In [ ]:
# Create Success Response Handler
def handle_success_response(response: requests.Response) -> Dict:
    """Manejar respuesta exitosa de creación masiva"""
    try:
        response_data = response.json()
        
        success_info = {
            "status": "success",
            "timestamp": datetime.now().isoformat(),
            "municipios_created": len(response_data.get("datos", [])),
            "message": response_data.get("mensaje", "Municipios creados exitosamente"),
            "data": response_data.get("datos", [])
        }
        
        return success_info
    except json.JSONDecodeError:
        return {
            "status": "success",
            "timestamp": datetime.now().isoformat(),
            "message": "Operación completada",
            "raw_response": response.text
        }

def send_bulk_municipios_request(base_url: str, auth_token: str, request_data: Dict) -> Dict:
    """Enviar request de creación masiva con manejo completo de respuesta"""
    
    # Validar antes de enviar
    is_valid, errors = validate_bulk_request(request_data)
    if not is_valid:
        return {
            "status": "validation_error",
            "errors": errors,
            "timestamp": datetime.now().isoformat()
        }
    
    # Configurar headers
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {auth_token}"
    }
    
    endpoint_url = f"{base_url}/api/catalog/municipios/bulk"
    
    try:
        # Enviar request
        logger.info(f"Enviando request a: {endpoint_url}")
        logger.info(f"Data: {json.dumps(request_data, indent=2)}")
        
        response = requests.post(
            endpoint_url,
            json=request_data,
            headers=headers,
            timeout=30
        )
        
        if response.status_code == 201:
            # Éxito
            return handle_success_response(response)
        else:
            # Error de la API
            error = parse_api_error_response(response)
            return {
                "status": "api_error",
                "error": {
                    "message": error.message,
                    "code": error.code,
                    "details": error.details
                },
                "timestamp": datetime.now().isoformat(),
                "status_code": response.status_code
            }
            
    except requests.exceptions.RequestException as e:
        return {
            "status": "network_error",
            "error": str(e),
            "timestamp": datetime.now().isoformat()
        }

# Ejemplo de uso completo
def demonstrate_complete_workflow():
    """Demostrar flujo completo de creación masiva"""
    print("🚀 DEMOSTRACIÓN DE FLUJO COMPLETO")
    print("="*60)
    
    # Configuración (simular)
    BASE_URL = "http://localhost:3000"  # O la URL de tu servidor
    AUTH_TOKEN = "your_jwt_token_here"
    
    # Datos de ejemplo corregidos
    municipios_ejemplo = [
        "Municipio Prueba 1",
        "Municipio Prueba 2", 
        {
            "nombre_municipio": "Municipio Completo",
            "codigo_dane": "99999",
            "id_departamento": 1
        }
    ]
    
    # Crear request correcto
    request_data = create_corrected_request_body(municipios_ejemplo, default_departamento_id=1)
    
    print("📤 Request final a enviar:")
    print(json.dumps(request_data, indent=2, ensure_ascii=False))
    
    print("\n✅ Validación:")
    is_valid, errors = validate_bulk_request(request_data)
    print(f"Request válido: {is_valid}")
    if errors:
        print("Errores:", errors)
    
    print("\n📋 Comando curl equivalente:")
    print(f"""
curl -X POST {BASE_URL}/api/catalog/municipios/bulk \\
  -H "Content-Type: application/json" \\
  -H "Authorization: Bearer {AUTH_TOKEN}" \\
  -d '{json.dumps(request_data, ensure_ascii=False)}'
    """)
    
    print("\n🔍 En Postman/Thunder Client:")
    print(f"POST {BASE_URL}/api/catalog/municipios/bulk")
    print("Headers:")
    print("  Content-Type: application/json")
    print(f"  Authorization: Bearer {AUTH_TOKEN}")
    print("Body (raw JSON):")
    print(json.dumps(request_data, indent=2, ensure_ascii=False))
    
    # Simular respuesta exitosa
    print("\n✅ Respuesta esperada (éxito):")
    expected_success = {
        "exito": True,
        "mensaje": "Municipios creados exitosamente",
        "datos": [
            {"id_municipio": 100, "nombre_municipio": "Municipio Prueba 1"},
            {"id_municipio": 101, "nombre_municipio": "Municipio Prueba 2"},
            {"id_municipio": 102, "nombre_municipio": "Municipio Completo"}
        ]
    }
    print(json.dumps(expected_success, indent=2, ensure_ascii=False))

demonstrate_complete_workflow()

# Resumen y Request Body Corregido

## ✅ Request Body Correcto para tu caso:

```json
{
  "municipios": [
    "string",
    {
      "nombre_municipio": "Bogotá D.C.",
      "codigo_dane": "11001",
      "id_departamento": 1
    }
  ],
  "defaultDepartamentoId": 1
}
```

## 🔑 Puntos Clave:

1. **defaultDepartamentoId es OBLIGATORIO** cuando hay strings en el array
2. **Strings simples** se convierten automáticamente a objetos con `nombre_municipio` y el `defaultDepartamentoId`
3. **Objetos completos** mantienen su `id_departamento` si lo tienen, o usan el default
4. **Validación robusta** previene errores de base de datos

## 🧪 Para probar en Postman/Thunder Client:

**URL:** `POST http://localhost:3000/api/catalog/municipios/bulk`

**Headers:**
- `Content-Type: application/json`
- `Authorization: Bearer YOUR_JWT_TOKEN`

**Body:** El JSON corregido mostrado arriba

## 📚 Documentación del endpoint:

- **Acepta:** Array mixto de strings y objetos
- **Requiere:** `defaultDepartamentoId` para strings
- **Valida:** Nombres no vacíos, departamentos existentes
- **Retorna:** Array de municipios creados con sus IDs